# 🔬 M.Tech Review Demo
## Parameter-Efficient Fine-Tuning of Small Language Models for Document Analysis

### The core comparison this project makes:

```
┌──────────────────────┐     ┌──────────────────────┐     ┌──────────────────────┐
│   SLM — base         │     │   SLM + LoRA          │     │   LLM — baseline     │
│   Qwen1.5-0.5B       │     │   Qwen1.5-0.5B        │     │   Qwen1.5-7B         │
│   zero-shot          │     │   PEFT fine-tuned ✓   │     │   zero-shot          │
│   (before tuning)    │     │   ← your contribution │     │   (compare against)  │
└──────────────────────┘     └──────────────────────┘     └──────────────────────┘
```

**Research question:** Can a PEFT-adapted 0.5B SLM match a 7B LLM on document tasks,
at a fraction of the compute cost?

---
**Notebook agenda:**
1. Environment check
2. Architecture diagram
3. Dataset inspection
4. Load SLM (Qwen1.5-0.5B)
5. Base SLM zero-shot inference
6. Attach LoRA (PEFT) + fine-tune
7. PEFT-SLM inference
8. Load LLM (Qwen1.5-7B) + inference
9. 3-way comparison table
10. Efficiency metrics
11. Experiment log
12. Next steps

## 0. Environment check

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
print(f'Python      : {sys.version}')
print(f'PyTorch     : {torch.__version__}')
cuda_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
print(f'CUDA device : {cuda_info}')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM        : {vram:.1f} GB')

import transformers, peft, datasets
print(f'Transformers: {transformers.__version__}')
print(f'PEFT        : {peft.__version__}')
print(f'Datasets    : {datasets.__version__}')

---
## 1. System Architecture

```
┌─────────────────────────┐    ┌──────────────────────────┐
│   Native text docs      │    │  Scanned / image docs    │
│   PDF, DOCX (text layer)│    │  Photos, scans, no text  │
└────────────┬────────────┘    └───────────┬──────────────┘
             │                             │
             ▼                             ▼
   Direct text extraction           OCR engine
   (pdfplumber, python-docx)        (Tesseract)
             │                             │
             └───────────┬─────────────────┘
                         ▼
              Unified text pipeline
              (tagged: native vs OCR source)
                         │
              ┌──────────┴───────────┐
              ▼                      ▼
       PEFT fine-tuning         LLM baseline
       Qwen1.5-0.5B + LoRA      Qwen1.5-7B (zero-shot)
              │                      │
              └──────────┬───────────┘
                         ▼
            3-way evaluation (EM, F1, ROUGE, latency)
            SLM-base  vs  PEFT-SLM  vs  LLM
```

---
## 2. Text Extraction Pipeline Demo

In [ ]:
from src.extraction.text_extractor import extract_text, chunk_text

# Plain text pass-through
r = extract_text('LoRA adds trainable low-rank matrices to frozen transformer layers.')
print(f'Source: {r["source"]}  |  Text: {r["text"]}')

# Chunking long documents
long_doc = 'word ' * 300
chunks = chunk_text(long_doc, max_tokens=100, overlap_tokens=10)
print(f'300-word doc → {len(chunks)} overlapping chunks of ~100 words each')

---
## 3. Dataset Inspection

In [ ]:
from datasets import load_dataset

print('=== SQuAD v2 — Question Answering ===')
squad = load_dataset('squad_v2', split='validation[:3]')
for ex in squad:
    ans = ex['answers']['text'][0] if ex['answers']['text'] else 'unanswerable'
    print(f"  Q: {ex['question']}")
    print(f"  A: {ans}")
    print(f"  Context: {ex['context'][:80]}...\n")

In [ ]:
print('=== XL-Sum English — Summarization ===')
xlsum = load_dataset('csebuetnlp/xlsum', 'english', split='validation[:3]')
for ex in xlsum:
    print(f"  Summary : {ex['summary']}")
    print(f"  Article : {ex['text'][:100]}...\n")

---
## 4. Load SLM — Qwen1.5-0.5B (500 M params)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SLM_MODEL = 'Qwen/Qwen1.5-0.5B'
LLM_MODEL = 'Qwen/Qwen1.5-7B'
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype     = torch.bfloat16 if device == 'cuda' else torch.float32

print(f'Loading SLM: {SLM_MODEL} on {device}...')
slm_tokenizer = AutoTokenizer.from_pretrained(SLM_MODEL, trust_remote_code=True)
if slm_tokenizer.pad_token is None:
    slm_tokenizer.pad_token = slm_tokenizer.eos_token

slm_model = AutoModelForCausalLM.from_pretrained(
    SLM_MODEL, trust_remote_code=True, torch_dtype=dtype, device_map=device
)
slm_model.config.use_cache = False

slm_params = sum(p.numel() for p in slm_model.parameters()) / 1e6
print(f'SLM loaded — {slm_params:.0f} M parameters')

---
## 5. Base SLM — Zero-Shot Inference (before any fine-tuning)

In [ ]:
import time

EXAMPLES = [
    {'context': 'The Eiffel Tower is located in Paris, France. It was built in 1889.',
     'question': 'Where is the Eiffel Tower located?', 'answer': 'Paris, France'},
    {'context': 'LoRA (Low-Rank Adaptation) is a PEFT technique that adds trainable low-rank matrices to transformer layers.',
     'question': 'What does LoRA stand for?', 'answer': 'Low-Rank Adaptation'},
    {'context': 'Python is a high-level programming language created by Guido van Rossum, first released in 1991.',
     'question': 'Who created Python?', 'answer': 'Guido van Rossum'},
]

def make_prompt(ctx, q):
    return f'### Context:\n{ctx}\n\n### Question:\n{q}\n\n### Answer:\n'

def generate(model, tok, prompt, max_new=40):
    inputs = tok(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                              pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    return tok.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

prompts = [make_prompt(e['context'], e['question']) for e in EXAMPLES]
answers = [e['answer'] for e in EXAMPLES]

print('BASE SLM (zero-shot):')
t0 = time.perf_counter()
base_preds = [generate(slm_model, slm_tokenizer, p) for p in prompts]
base_lat   = (time.perf_counter() - t0) * 1000
for ex, pred in zip(EXAMPLES, base_preds):
    print(f"  Q: {ex['question']}")
    print(f"  Ref : {ex['answer']}")
    print(f"  Pred: {pred}\n")

---
## 6. Attach LoRA (PEFT) + Fine-Tune

In [ ]:
from peft import LoraConfig, get_peft_model
import torch.optim as optim

lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none', task_type='CAUSAL_LM',
)
peft_model = get_peft_model(slm_model, lora_config)

trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in peft_model.parameters())
print(f'LoRA attached — trainable: {trainable/1e3:.1f}K / {total/1e6:.0f}M params ({100*trainable/total:.3f}%)')

# Build full prompt+answer for training
full_texts = [make_prompt(e['context'], e['question']) + e['answer'] + slm_tokenizer.eos_token
              for e in EXAMPLES]

optimizer = optim.AdamW(peft_model.parameters(), lr=2e-4)
peft_model.train()
STEPS = 5  # increase to 50+ for real training

t0 = time.perf_counter()
for step in range(STEPS):
    step_loss = 0.0
    for ft in full_texts:
        inp = slm_tokenizer(ft, return_tensors='pt', truncation=True, max_length=256).to(device)
        out = peft_model(**inp, labels=inp['input_ids'].clone())
        out.loss.backward()
        step_loss += out.loss.item()
    optimizer.step(); optimizer.zero_grad()
    print(f'  Step {step+1}/{STEPS} | avg loss: {step_loss/len(full_texts):.4f}')

train_time = time.perf_counter() - t0
print(f'\nLoRA fine-tuning done in {train_time:.1f}s')

---
## 7. PEFT-SLM Inference ← your main contribution

In [ ]:
peft_model.eval()
print('PEFT-SLM (LoRA fine-tuned):')
t0 = time.perf_counter()
peft_preds = [generate(peft_model, slm_tokenizer, p) for p in prompts]
peft_lat   = (time.perf_counter() - t0) * 1000
for ex, pred in zip(EXAMPLES, peft_preds):
    print(f"  Q: {ex['question']}")
    print(f"  Ref : {ex['answer']}")
    print(f"  Pred: {pred}\n")

---
## 8. Load LLM — Qwen1.5-7B and run zero-shot inference

In [ ]:
# NOTE: Qwen1.5-7B needs ~14 GB VRAM (bfloat16) or ~7 GB (4-bit QLoRA).
# If you don't have enough VRAM, set use_4bit=True below.
use_4bit = False  # ← set True on low-VRAM machines

if use_4bit:
    from transformers import BitsAndBytesConfig
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL, trust_remote_code=True, quantization_config=bnb_cfg, device_map=device
    )
else:
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL, trust_remote_code=True, torch_dtype=dtype, device_map=device
    )

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

llm_params = sum(p.numel() for p in llm_model.parameters()) / 1e9
print(f'LLM loaded — {llm_params:.1f} B parameters (zero-shot, no fine-tuning)')

print('\nLLM (Qwen1.5-7B, zero-shot):')
t0 = time.perf_counter()
llm_preds = [generate(llm_model, llm_tokenizer, p) for p in prompts]
llm_lat   = (time.perf_counter() - t0) * 1000
for ex, pred in zip(EXAMPLES, llm_preds):
    print(f"  Q: {ex['question']}")
    print(f"  Ref : {ex['answer']}")
    print(f"  Pred: {pred}\n")

---
## 9. 3-Way Comparison Table

In [ ]:
from src.evaluation.metrics import compute_qa_metrics

base_m = compute_qa_metrics(base_preds, answers)
peft_m = compute_qa_metrics(peft_preds, answers)
llm_m  = compute_qa_metrics(llm_preds,  answers)

print('=' * 72)
print('  3-WAY COMPARISON  |  Task: QA  |  Dataset: demo examples')
print('=' * 72)
print(f'{"Model / Method":<30} {"EM":>8} {"F1":>8} {"Latency(ms)":>14} {"Params":>8}')
print('-' * 72)
print(f'{"SLM — base (zero-shot)":<30} {base_m["exact_match"]:>8.1f} {base_m["f1"]:>8.1f} {base_lat:>14.0f} {"0.5B":>8}')
print(f'{"SLM + LoRA  (PEFT-tuned)":<30} {peft_m["exact_match"]:>8.1f} {peft_m["f1"]:>8.1f} {peft_lat:>14.0f} {"0.5B":>8}')
print(f'{"LLM — Qwen1.5-7B (zero-shot)":<30} {llm_m["exact_match"]:>8.1f} {llm_m["f1"]:>8.1f} {llm_lat:>14.0f} {"7B":>8}')
print('=' * 72)
print('  ↑  Key result: PEFT-SLM row should approach LLM row at 0.5B cost.')
print('=' * 72)

---
## 10. Efficiency Comparison

In [ ]:
import matplotlib.pyplot as plt

labels   = ['SLM base\n(0.5B)', 'PEFT-SLM\n(0.5B + LoRA)', 'LLM\n(7B)']
f1_vals  = [base_m['f1'], peft_m['f1'], llm_m['f1']]
lat_vals = [base_lat, peft_lat, llm_lat]
params   = [0.5, 0.5, 7.0]   # billions

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = ['#4e79a7', '#f28e2b', '#e15759']

axes[0].bar(labels, f1_vals, color=colors)
axes[0].set_title('F1 Score (higher = better)')
axes[0].set_ylim(0, 100)
axes[0].set_ylabel('F1')

axes[1].bar(labels, lat_vals, color=colors)
axes[1].set_title('Inference Latency ms (lower = better)')
axes[1].set_ylabel('ms')

axes[2].bar(labels, params, color=colors)
axes[2].set_title('Model Size (lower = more efficient)')
axes[2].set_ylabel('Billion parameters')

plt.tight_layout()
plt.savefig('../experiments/comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to experiments/comparison_chart.png')

---
## 11. Experiment Log

In [ ]:
from src.utils.experiment_tracker import ExperimentTracker

tracker = ExperimentTracker('../experiments/results.csv')

tracker.log({'experiment': 'nb_base_slm', 'model': SLM_MODEL, 'dataset': 'demo_qa',
             'task': 'qa', 'method': 'base_SLM', 'test_samples': len(EXAMPLES),
             'exact_match': base_m['exact_match'], 'f1': base_m['f1'],
             'avg_latency_ms': round(base_lat/len(EXAMPLES), 1)})

tracker.log({'experiment': 'nb_peft_slm', 'model': SLM_MODEL, 'dataset': 'demo_qa',
             'task': 'qa', 'method': 'PEFT_LoRA', 'test_samples': len(EXAMPLES),
             'exact_match': peft_m['exact_match'], 'f1': peft_m['f1'],
             'avg_latency_ms': round(peft_lat/len(EXAMPLES), 1),
             'train_time_s': round(train_time, 1)})

tracker.log({'experiment': 'nb_llm_baseline', 'model': LLM_MODEL, 'dataset': 'demo_qa',
             'task': 'qa', 'method': 'LLM_baseline', 'test_samples': len(EXAMPLES),
             'exact_match': llm_m['exact_match'], 'f1': llm_m['f1'],
             'avg_latency_ms': round(llm_lat/len(EXAMPLES), 1)})

df = tracker.load()
display(df[['experiment','model','method','exact_match','f1','avg_latency_ms','train_time_s']])

---
## 12. Next Steps

| Step | Action |
|---|---|
| ✅ Done | Repository scaffold, pipeline, demo, 3-way comparison framework |
| Now | Run `demo_quick.py` on a machine with GPU |
| Week 1 | Baseline eval — SLM-base on 500 SQuAD test examples |
| Week 2 | Full LoRA fine-tuning — 3 epochs, 5000 SQuAD train examples |
| Week 3 | LLM baseline eval — Qwen1.5-7B on same 500 test examples |
| Week 4 | Compile 3-way results table; write Phase 1 report |
| Sem 4 | Add summarization (XL-Sum), extraction (FUNSD), classification (RVL-CDIP) |
| Sem 4 | OCR experiments — DocVQA scanned docs |
| Sem 4 | Error analysis + thesis writing |

---

### The 3-way comparison — what each result tells us

| Comparison | What it answers |
|---|---|
| PEFT-SLM vs Base-SLM | Does LoRA fine-tuning actually help the small model? |
| PEFT-SLM vs LLM | Can a tuned small model compete with a much larger one? |
| Latency: PEFT-SLM vs LLM | How much faster / cheaper is the small model to run? |

**Main research claim to validate:**
> *PEFT-adapted SLMs can approach LLM-level performance on selected document tasks
> while requiring significantly less compute.*

---
*M.Tech Project — A Study on Parameter-Efficient Fine-Tuning of Small Language Models for Document Analysis*